## Loading BLIP model

In [1]:
import torch
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


## Caption ONE keyframe

In [2]:
def caption_image(image_path):
    image = Image.open(image_path).convert("RGB")

    inputs = processor(image, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=40,
            num_beams=5
        )

    caption = processor.decode(output[0], skip_special_tokens=True)
    return caption


## Caption all shots of ONE video

In [3]:
import os
import json

def caption_video_keyframes(video_folder_path, output_json):
    keyframes_dir = os.path.join(video_folder_path, "keyframes")

    if not os.path.isdir(keyframes_dir):
        print(f"[WARN] No keyframes in {video_folder_path}")
        return

    captions = {}

    for fname in sorted(os.listdir(keyframes_dir)):
        if not fname.lower().endswith(".jpg"):
            continue

        shot_id = int(fname.split("_")[1].split(".")[0])
        img_path = os.path.join(keyframes_dir, fname)

        caption = caption_image(img_path)
        captions[shot_id] = caption

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(captions, f, indent=2)

    return captions


## Batch captioning over all videos

In [4]:
SEGMENTATION_DIR = "../data/segmentation"
CAPTIONS_DIR = "../data/captions"

os.makedirs(CAPTIONS_DIR, exist_ok=True)

for video_name in os.listdir(SEGMENTATION_DIR):
    video_folder = os.path.join(SEGMENTATION_DIR, video_name)
    if not os.path.isdir(video_folder):
        continue

    out_json = os.path.join(CAPTIONS_DIR, f"{video_name}.json")

    if os.path.exists(out_json):
        print(f"[SKIP] Captions exist for {video_name}")
        continue

    print(f"[CAPTION] Processing {video_name}")
    caption_video_keyframes(video_folder, out_json)

print("✅ Visual captioning complete")


[CAPTION] Processing 16 0613 PROJ COLOR CHEF
[CAPTION] Processing 6607010_2158732_DE_de_9000D_Winter Sale_Feed&Masthead_YT_Women_Step2_16x9_20s_D - H&M (720p, h264, youtube)
[CAPTION] Processing Adidas _Beyond The Blue_ AI Commercial (Midjourney + @RunwayML + @topazlabs )  AI Advertising
[CAPTION] Processing ad_01
[CAPTION] Processing All the Best Moments are Better With Pepsi
[CAPTION] Processing Banned M&M's Commercial
[CAPTION] Processing BORZ WEAR (Streetwear Commercial Video) - ICE SQUAD MEDIA
[CAPTION] Processing CINEMATIC PRODUCT VIDEO _ Go Good Drinks
[CAPTION] Processing Clothing commercial video _ Jacferdi _ Fujifilm X-T3
[CAPTION] Processing Essentials Clothing Shoot _ Promotional Video
[CAPTION] Processing Every Table Has A Story
[CAPTION] Processing Gucci Presents_ Gucci Oud, the new unisex fragrance
[CAPTION] Processing Introducting ACQUA DI GIÒ PROFONDO PARFUM by Giorgio Armani
[CAPTION] Processing Kellogg’s Multigrain Chocos _ Multigrain Energy, More Chocolatey _ Hindi 